# NumPy & Pandas — The ML Data Stack

## What This Notebook Covers

`python_for_ml.ipynb` gave you the fundamentals.
This notebook goes deeper — the operations you actually need daily.

| Section | Topic |
|---|---|
| 1 | NumPy: Stacking, Splitting & Saving Arrays |
| 2 | NumPy: Advanced Indexing & Boolean Operations |
| 3 | Pandas: Loading Real Data & First Inspection |
| 4 | Pandas: Merging & Joining (SQL JOIN in Python) |
| 5 | Pandas: GroupBy & Aggregation |
| 6 | Pandas: Apply, Map & Custom Transforms |
| 7 | Pandas: Reshaping — Melt & Pivot |
| 8 | The NumPy-Pandas Bridge |

## The Mental Model

```
Raw CSV / Database
       |
  pd.DataFrame    <- column names, mixed types, NaN tracking, SQL-like ops
       |
  df.values  /  df.to_numpy()
       |
  np.ndarray      <- pure numbers, vectorized math, sklearn/torch input
       |
  sklearn / PyTorch
```


In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 100)
pd.set_option('display.float_format', '{:.3f}'.format)

print(f"NumPy  {np.__version__}")
print(f"Pandas {pd.__version__}")


NumPy  2.4.2
Pandas 3.0.1


---
## Section 1 — NumPy: Stacking, Splitting & Saving

### Why This Matters

In a real ML pipeline you constantly:
- **Stack** arrays from different sources into one training matrix
- **Split** a matrix into feature groups for separate preprocessing
- **Save / load** expensive computed arrays to disk

### Stacking Reference

```python
np.vstack([a, b])      # vertical stack   — stack rows   (axis 0)
np.hstack([a, b])      # horizontal stack — stack cols   (axis 1)
np.concatenate([a, b], axis=0)   # general, explicit axis
np.column_stack([v1, v2, v3])    # stack 1D vectors as columns
```


In [2]:
import tempfile
import os

print("=== Stacking Arrays ===")
print()

np.random.seed(42)
n = 100

# Simulate feature groups from different sources
numeric_feats  = np.random.randn(n, 5)      # 5 continuous features
encoded_cats   = np.eye(3)[np.random.randint(0, 3, n)]  # 3 one-hot cols
interaction    = (numeric_feats[:, 0] * numeric_feats[:, 1]).reshape(-1, 1)

# Stack horizontally into one feature matrix
X = np.hstack([numeric_feats, encoded_cats, interaction])
print(f"  numeric_feats:  {numeric_feats.shape}")
print(f"  encoded_cats:   {encoded_cats.shape}")
print(f"  interaction:    {interaction.shape}")
print(f"  X (combined):   {X.shape}   <- final feature matrix")
print()

# vstack: add new training data
X_batch1 = np.random.randn(50, 5)
X_batch2 = np.random.randn(30, 5)
X_all    = np.vstack([X_batch1, X_batch2])
print(f"  vstack: {X_batch1.shape} + {X_batch2.shape} = {X_all.shape}")
print()

# column_stack: build from 1D vectors
ages     = np.array([25, 35, 45])
incomes  = np.array([40000, 70000, 90000])
scores   = np.array([0.7, 0.8, 0.9])
table    = np.column_stack([ages, incomes, scores])
print(f"  column_stack: {table}")
print()

print("=== Splitting Arrays ===")
print()

X_data = np.arange(40).reshape(8, 5)
y_data = np.arange(8)

# Split along rows (axis 0) — for train/val/test
n_total = len(X_data)
n_train = int(0.6 * n_total)
n_val   = int(0.2 * n_total)

idx     = np.random.permutation(n_total)
X_tr = X_data[idx[:n_train]]
X_va = X_data[idx[n_train:n_train+n_val]]
X_te = X_data[idx[n_train+n_val:]]
print(f"  Manual split:  train={X_tr.shape}  val={X_va.shape}  test={X_te.shape}")

# np.split — equal splits
parts = np.split(X_data, [5, 7], axis=0)   # rows 0-4, 5-6, 7
print(f"  np.split rows: {[p.shape for p in parts]}")

# Split along columns — separate feature groups
X_num, X_cat = np.split(X_data, [3], axis=1)  # first 3 cols | last 2 cols
print(f"  Column split:  numeric={X_num.shape}  categorical={X_cat.shape}")
print()

print("=== Saving & Loading Arrays ===")
print()

# .npy — single array, fastest
arr = np.random.randn(1000, 50)
with tempfile.NamedTemporaryFile(suffix='.npy', delete=False) as f:
    npy_path = f.name
np.save(npy_path, arr)
loaded = np.load(npy_path)
print(f"  .npy  save/load: shape={loaded.shape}  match={np.allclose(arr, loaded)}")
os.unlink(npy_path)

# .npz — multiple arrays in one file (compressed)
X_arr = np.random.randn(500, 20)
y_arr = np.random.randint(0, 2, 500)
with tempfile.NamedTemporaryFile(suffix='.npz', delete=False) as f:
    npz_path = f.name
np.savez(npz_path, X=X_arr, y=y_arr)
data = np.load(npz_path)
print(f"  .npz  keys: {list(data.keys())}  X={data['X'].shape}  y={data['y'].shape}")
os.unlink(npz_path)

print()
print("  When to use each format:")
print("  .npy   -- single large array (embeddings, preprocessed features)")
print("  .npz   -- multiple arrays together (X_train, X_test, y_train, y_test)")
print("  .pkl   -- sklearn models (via joblib)")
print("  .csv   -- human-readable small data (slow for large datasets)")
print("  .parquet -- large tabular data with Pandas (fast, columnar, compressed)")


=== Stacking Arrays ===

  numeric_feats:  (100, 5)
  encoded_cats:   (100, 3)
  interaction:    (100, 1)
  X (combined):   (100, 9)   <- final feature matrix

  vstack: (50, 5) + (30, 5) = (80, 5)

  column_stack: [[2.5e+01 4.0e+04 7.0e-01]
 [3.5e+01 7.0e+04 8.0e-01]
 [4.5e+01 9.0e+04 9.0e-01]]

=== Splitting Arrays ===

  Manual split:  train=(4, 5)  val=(1, 5)  test=(3, 5)
  np.split rows: [(5, 5), (2, 5), (1, 5)]
  Column split:  numeric=(8, 3)  categorical=(8, 2)

=== Saving & Loading Arrays ===

  .npy  save/load: shape=(1000, 50)  match=True
  .npz  keys: ['X', 'y']  X=(500, 20)  y=(500,)

  When to use each format:
  .npy   -- single large array (embeddings, preprocessed features)
  .npz   -- multiple arrays together (X_train, X_test, y_train, y_test)
  .pkl   -- sklearn models (via joblib)
  .csv   -- human-readable small data (slow for large datasets)
  .parquet -- large tabular data with Pandas (fast, columnar, compressed)


---
## Section 2 — NumPy: Advanced Indexing & Operations

### Fancy Indexing — Select Any Rows/Cols by Index

```python
rows = [0, 3, 7]
X[rows]          # rows 0, 3, 7
X[rows, :]       # same
X[:, [1, 4]]     # columns 1 and 4
X[rows, :][:, [1, 4]]  # both: rows 0,3,7 AND columns 1,4
```

### Boolean Masking at Scale

```python
# Pick samples where feature 2 > 0 AND label == 1
mask = (X[:, 2] > 0) & (y == 1)
X[mask], y[mask]
```

### Vectorized Comparisons — no for-loops


In [3]:
np.random.seed(42)
n_samples, n_feats = 200, 8

X = np.random.randn(n_samples, n_feats)
y = (X[:, 0] + X[:, 1] > 0).astype(int)   # synthetic binary label

print("=== Fancy Indexing ===")
print()

# Select specific rows
top_rows   = np.argsort(X[:, 0])[-5:]   # 5 rows with highest feature-0
print(f"  Top-5 rows by feature 0: {top_rows}")
print(f"  Their feature values:    {X[top_rows, 0].round(3)}")
print()

# Select specific columns
key_cols  = [0, 3, 7]
X_sub     = X[:, key_cols]
print(f"  Columns {key_cols}: shape {X_sub.shape}")
print()

# 2D fancy indexing — pick specific (row, col) pairs
rows = [0, 10, 20, 30]
cols = [1,  3,  5,  7]
vals = X[rows, cols]   # picks X[0,1], X[10,3], X[20,5], X[30,7]
print(f"  Diagonal-style indexing: {vals.round(3)}")
print()

print("=== Boolean Masking ===")
print()

pos_class  = y == 1
neg_class  = y == 0
print(f"  Positive samples: {pos_class.sum()}   Negative: {neg_class.sum()}")
print(f"  Positive mean f0: {X[pos_class, 0].mean():.3f}")
print(f"  Negative mean f0: {X[neg_class, 0].mean():.3f}")
print()

# Multi-condition mask
extreme_pos = (X[:, 0] > 1.5) & (X[:, 1] > 1.5) & (y == 1)
print(f"  Extreme positives (f0>1.5 AND f1>1.5 AND y=1): {extreme_pos.sum()} samples")
print()

print("=== np.where — Conditional Element Selection ===")
print()

# Vectorized ternary: np.where(condition, value_if_true, value_if_false)
relu_X   = np.where(X > 0, X, 0)           # ReLU activation
clipped  = np.where(X > 2, 2, np.where(X < -2, -2, X))   # clip to [-2, 2]
sign_X   = np.where(X > 0, 1, -1)          # sign function

print(f"  X range:       [{X.min():.2f}, {X.max():.2f}]")
print(f"  ReLU range:    [{relu_X.min():.2f}, {relu_X.max():.2f}]")
print(f"  Clipped range: [{clipped.min():.2f}, {clipped.max():.2f}]")
print()

print("=== Vectorized Statistics Per Class ===")
print()

# Compute per-class statistics without a loop
for cls in [0, 1]:
    mask  = y == cls
    Xc    = X[mask]
    print(f"  Class {cls}:  n={mask.sum():3d}  "
          f"mean_f0={Xc[:, 0].mean():+.3f}  "
          f"std_f0={Xc[:, 0].std():.3f}  "
          f"max_f0={Xc[:, 0].max():.3f}")
print()

print("=== Broadcasting: Standardize Per-Feature ===")
print()
# Manual StandardScaler — shows exactly what sklearn does
mean = X.mean(axis=0)   # shape (8,)
std  = X.std(axis=0)    # shape (8,)
X_scaled = (X - mean) / std   # broadcasting: (200, 8) - (8,) works!

print(f"  Before scaling: mean={X.mean(axis=0)[:3].round(2)}  std={X.std(axis=0)[:3].round(2)}")
print(f"  After scaling:  mean={X_scaled.mean(axis=0)[:3].round(10)}  std={X_scaled.std(axis=0)[:3].round(2)}")
print()
print("  That 3-line manual scaler IS exactly what StandardScaler does internally.")


=== Fancy Indexing ===

  Top-5 rows by feature 0: [ 31 118 174 187 110]
  Their feature values:    [1.765 1.847 2.013 2.062 2.527]

  Columns [0, 3, 7]: shape (200, 3)

  Diagonal-style indexing: [-0.138 -0.518  0.413 -0.653]

=== Boolean Masking ===

  Positive samples: 96   Negative: 104
  Positive mean f0: 0.530
  Negative mean f0: -0.486

  Extreme positives (f0>1.5 AND f1>1.5 AND y=1): 0 samples

=== np.where — Conditional Element Selection ===

  X range:       [-3.24, 3.85]
  ReLU range:    [0.00, 3.85]
  Clipped range: [-2.00, 2.00]

=== Vectorized Statistics Per Class ===

  Class 0:  n=104  mean_f0=-0.486  std_f0=0.691  max_f0=1.189
  Class 1:  n= 96  mean_f0=+0.530  std_f0=0.765  max_f0=2.527

=== Broadcasting: Standardize Per-Feature ===

  Before scaling: mean=[0.   0.07 0.06]  std=[0.89 0.97 0.98]
  After scaling:  mean=[-0. -0. -0.]  std=[1. 1. 1.]

  That 3-line manual scaler IS exactly what StandardScaler does internally.


---
## Section 3 — Pandas: Loading Real Data & First Inspection

### The First 5 Commands on Any New Dataset

Every time you get a new dataset, run these before anything else:

```python
df = pd.read_csv('data.csv')

df.shape          # how big is it?
df.dtypes         # what types got inferred?
df.describe()     # stats on numeric columns
df.isnull().sum() # how many missing per column?
df.head()         # eyeball the data
```

These 5 lines tell you 90% of what you need to know before cleaning.

### Common `read_csv` Options

```python
pd.read_csv('data.csv', index_col='Id')          # set ID as row index
pd.read_csv('data.csv', usecols=['a','b','c'])    # load only some columns
pd.read_csv('data.csv', dtype={'col': np.float32}) # force dtype
pd.read_csv('data.csv', na_values=['?', 'N/A'])  # extra NaN markers
pd.read_csv('data.csv', nrows=1000)              # preview first 1000 rows
```


In [4]:
np.random.seed(42)
print("=== Generating Realistic Dataset ===")
print()

# Simulate a house prices dataset (same structure as Kaggle)
n = 300
neighborhoods = ['North', 'South', 'East', 'West', 'Central']

df = pd.DataFrame({
    'Id':          np.arange(1, n + 1),
    'GrLivArea':   np.random.normal(1500, 400, n).clip(400, 4000).round(0),
    'OverallQual': np.random.randint(1, 11, n),
    'YearBuilt':   np.random.randint(1950, 2024, n),
    'GarageCars':  np.random.choice([0, 1, 2, 3], n, p=[0.05, 0.20, 0.60, 0.15]),
    'TotalBsmtSF': np.random.normal(1000, 300, n).clip(0, 3000).round(0),
    'Neighborhood':np.random.choice(neighborhoods, n),
    'BldgType':    np.random.choice(['1Fam', 'TwnhsE', 'Duplex'], n, p=[0.7, 0.2, 0.1]),
    'SalePrice':   None,
})

# Correlated sale price
df['SalePrice'] = (
    df['GrLivArea'] * 80 +
    df['OverallQual'] * 8000 +
    (df['YearBuilt'] - 1950) * 400 +
    df['GarageCars'] * 5000 +
    np.random.normal(0, 15000, n)
).clip(50000, 800000).round(-3)

# Introduce realistic messiness
missing_idx = np.random.choice(n, 30, replace=False)
df.loc[missing_idx[:15], 'GrLivArea']   = np.nan
df.loc[missing_idx[15:], 'TotalBsmtSF'] = np.nan
df.loc[np.random.choice(n, 5), 'GarageCars'] = np.nan

tmp = tempfile.mktemp(suffix='.csv')
df.to_csv(tmp, index=False)

print("=== Loading & First Inspection ===")
print()
df = pd.read_csv(tmp, index_col='Id')
os.unlink(tmp)

print(f"Shape:   {df.shape}")
print()
print("dtypes:")
print(df.dtypes)
print()

print("describe() — numeric columns:")
print(df.describe().round(1))
print()

print("Missing values per column:")
missing = df.isnull().sum()
print(missing[missing > 0])
print()

print("head(3):")
print(df.head(3))
print()

# Value counts for categoricals
print("Categorical distributions:")
for col in ['Neighborhood', 'BldgType', 'GarageCars']:
    vc = df[col].value_counts()
    print(f"  {col}:  {dict(vc)}")


=== Generating Realistic Dataset ===

=== Loading & First Inspection ===

Shape:   (300, 8)

dtypes:
GrLivArea       float64
OverallQual       int64
YearBuilt         int64
GarageCars      float64
TotalBsmtSF     float64
Neighborhood        str
BldgType            str
SalePrice       float64
dtype: object

describe() — numeric columns:
       GrLivArea  OverallQual  YearBuilt  GarageCars  TotalBsmtSF  SalePrice
count    285.000      300.000    300.000     295.000      285.000    300.000
mean    1500.200        5.400   1985.500       1.900     1026.100 186650.000
std      397.700        2.900     21.500       0.800      323.300  40427.000
min      400.000        1.000   1950.000       0.000      263.000  75000.000
25%     1223.000        3.000   1967.000       2.000      826.000 157000.000
50%     1526.000        5.000   1985.500       2.000     1028.000 182500.000
75%     1754.000        8.000   2004.000       2.000     1242.000 214000.000
max     3041.000       10.000   2023.000      

---
## Section 4 — Pandas: Merging & Joining

### SQL JOIN in Python

If you know SQL, this is immediately familiar.
Every Kaggle competition and real project involves joining multiple tables.

```
LEFT JOIN   pd.merge(left, right, how='left',  on='key')
INNER JOIN  pd.merge(left, right, how='inner', on='key')
OUTER JOIN  pd.merge(left, right, how='outer', on='key')
RIGHT JOIN  pd.merge(left, right, how='right', on='key')
```

### When You Need This

- Houses table + neighborhood features table (join on `Neighborhood`)
- Customers table + purchase history table (join on `customer_id`)
- `train.csv` + supplemental feature CSV (join on `Id`)
- Cross-reference train IDs with external label file


In [5]:
np.random.seed(42)
print("=== Building Two Tables to Join ===")
print()

# Table 1: house features (like train.csv)
houses = pd.DataFrame({
    'Id':          [1, 2, 3, 4, 5, 6, 7],
    'Neighborhood':['North','South','East','North','West','East','South'],
    'GrLivArea':   [1500, 2000, 1200, 1800, 900, 2200, 1600],
    'SalePrice':   [220000, 350000, 155000, 280000, 120000, 380000, 240000],
})

# Table 2: neighborhood statistics (supplemental data)
nbhd_stats = pd.DataFrame({
    'Neighborhood': ['North', 'South', 'East', 'West'],
    'MedianIncome': [75000, 95000, 55000, 45000],
    'CrimeIndex':   [2.1, 1.4, 3.8, 4.5],
    'SchoolRating': [8, 9, 6, 5],
})

print("Houses table:")
print(houses)
print()
print("Neighborhood stats table:")
print(nbhd_stats)
print()

# INNER JOIN — only rows with matching key in BOTH tables
inner = pd.merge(houses, nbhd_stats, on='Neighborhood', how='inner')
print(f"INNER JOIN (7 houses, 4 neighborhoods with stats): {len(inner)} rows")
print(inner[['Id', 'Neighborhood', 'GrLivArea', 'MedianIncome', 'SchoolRating']])
print()

# LEFT JOIN — all houses, fill NaN where no match
left = pd.merge(houses, nbhd_stats, on='Neighborhood', how='left')
print(f"LEFT JOIN (keep all {len(houses)} houses): {len(left)} rows")
print(f"  Unmatched 'West' MedianIncome: {left.loc[left['Neighborhood']=='West','MedianIncome'].values}")
print()

# Joining on different column names
sales = pd.DataFrame({
    'house_id': [1, 3, 5, 7],
    'SaleDate': ['2023-01', '2023-03', '2023-02', '2023-04'],
    'AgentId':  [101, 102, 101, 103],
})
merged = pd.merge(houses, sales, left_on='Id', right_on='house_id', how='left')
print("Left join with different column names (Id vs house_id):")
print(merged[['Id', 'Neighborhood', 'SalePrice', 'SaleDate']].head(4))
print()

# concat — stacking DataFrames (same columns, more rows)
# Like UNION in SQL
train_q1 = houses.iloc[:4].copy()
train_q2 = houses.iloc[4:].copy()
all_data  = pd.concat([train_q1, train_q2], ignore_index=True)
print(f"pd.concat (stacking rows): {train_q1.shape} + {train_q2.shape} = {all_data.shape}")
print()

# concat — adding feature columns (same rows, more cols)
extra_feats = pd.DataFrame({
    'GarageArea': [400, 600, 0, 500, 200, 700, 450],
    'PoolArea':   [0, 100, 0, 0, 50, 200, 0],
}, index=houses['Id'])
houses_idx = houses.set_index('Id')
enriched   = pd.concat([houses_idx, extra_feats], axis=1)  # axis=1 = add columns
print(f"pd.concat (adding columns): {houses_idx.shape} + {extra_feats.shape} = {enriched.shape}")
print(enriched.head(3))


=== Building Two Tables to Join ===

Houses table:
   Id Neighborhood  GrLivArea  SalePrice
0   1        North       1500     220000
1   2        South       2000     350000
2   3         East       1200     155000
3   4        North       1800     280000
4   5         West        900     120000
5   6         East       2200     380000
6   7        South       1600     240000

Neighborhood stats table:
  Neighborhood  MedianIncome  CrimeIndex  SchoolRating
0        North         75000       2.100             8
1        South         95000       1.400             9
2         East         55000       3.800             6
3         West         45000       4.500             5

INNER JOIN (7 houses, 4 neighborhoods with stats): 7 rows
   Id Neighborhood  GrLivArea  MedianIncome  SchoolRating
0   1        North       1500         75000             8
1   2        South       2000         95000             9
2   3         East       1200         55000             6
3   4        North       180

---
## Section 5 — Pandas: GroupBy & Aggregation

### SQL `GROUP BY` in Python

```sql
-- SQL
SELECT Neighborhood, AVG(SalePrice), COUNT(*), STDDEV(SalePrice)
FROM houses
GROUP BY Neighborhood
```

```python
# Pandas — same operation
df.groupby('Neighborhood').agg(
    mean_price  = ('SalePrice', 'mean'),
    count       = ('SalePrice', 'count'),
    std_price   = ('SalePrice', 'std'),
)
```

### Why GroupBy Matters for Feature Engineering

Group statistics become powerful **target-encoded features**:
- `neighborhood_mean_price` — how expensive is this neighborhood on average?
- `neighborhood_std_price` — how variable are prices here?
- These features often rank in top importances for tree models.


In [6]:
np.random.seed(42)

# Realistic house dataset
n = 300
df = pd.DataFrame({
    'Neighborhood': np.random.choice(['Affluent','Midtown','Suburbs','Budget'], n, p=[0.2,0.3,0.35,0.15]),
    'BldgType':     np.random.choice(['1Fam','TwnhsE','Duplex'], n, p=[0.7,0.2,0.1]),
    'OverallQual':  np.random.randint(3, 11, n),
    'GrLivArea':    np.random.normal(1500, 400, n).clip(500, 4000).round(0),
    'YearBuilt':    np.random.randint(1960, 2020, n),
})
df['SalePrice'] = (
    df['GrLivArea'] * 85 +
    df['OverallQual'] * 9000 +
    df['Neighborhood'].map({'Affluent': 80000, 'Midtown': 40000, 'Suburbs': 10000, 'Budget': -30000}) +
    np.random.normal(0, 12000, n)
).clip(60000, 750000).round(-3)
df.loc[np.random.choice(n, 20, replace=False), 'SalePrice'] = np.nan

print("=== Basic GroupBy ===")
print()

# Single aggregation
nbhd_means = df.groupby('Neighborhood')['SalePrice'].mean().round(0)
print("Mean SalePrice by Neighborhood:")
print(nbhd_means.sort_values(ascending=False))
print()

# Multiple aggregations — the .agg() pattern
summary = (df.groupby('Neighborhood')
           .agg(
               count       = ('SalePrice', 'count'),
               mean_price  = ('SalePrice', 'mean'),
               median_price= ('SalePrice', 'median'),
               std_price   = ('SalePrice', 'std'),
               mean_size   = ('GrLivArea', 'mean'),
               mean_qual   = ('OverallQual', 'mean'),
           )
           .round(1)
           .sort_values('mean_price', ascending=False))

print("Full neighborhood summary:")
print(summary)
print()

# Two-level groupby
print("=== Multi-Level GroupBy ===")
print()
two_level = (df.groupby(['Neighborhood', 'BldgType'])
             .agg(count=('SalePrice','count'), mean_price=('SalePrice','mean'))
             .round(0)
             .sort_values('mean_price', ascending=False))
print(two_level.head(8))
print()

print("=== Feature Engineering with GroupBy ===")
print()
# Target encoding — map each row to its neighborhood's mean price
# CRITICAL: in production, compute this on TRAIN set only, then merge into test
# (to avoid data leakage)

nbhd_stats = (df.groupby('Neighborhood')['SalePrice']
              .agg(['mean', 'std', 'count'])
              .rename(columns={'mean': 'nbhd_mean_price',
                              'std':  'nbhd_std_price',
                              'count':'nbhd_count'})
              .round(1))

df_enriched = df.merge(nbhd_stats, on='Neighborhood', how='left')
print("After adding neighborhood group features:")
print(df_enriched[['Neighborhood', 'SalePrice', 'nbhd_mean_price', 'nbhd_std_price']].head(6))
print()
print("  New columns: nbhd_mean_price, nbhd_std_price, nbhd_count")
print(f"  Shape: {df.shape} -> {df_enriched.shape}")
print()

print("=== Transform — Apply Group Stats Row-by-Row ===")
print()
# transform() returns a Series aligned with the original DataFrame
# groupby + transform = add group stats as a new column directly
df['nbhd_mean']  = df.groupby('Neighborhood')['SalePrice'].transform('mean')
df['price_vs_nbhd'] = df['SalePrice'] / df['nbhd_mean']   # how much above/below average?

print("Price relative to neighborhood average:")
print(df[['Neighborhood','SalePrice','nbhd_mean','price_vs_nbhd']].head(8).round(2))


=== Basic GroupBy ===

Mean SalePrice by Neighborhood:
Neighborhood
Affluent   269115.000
Midtown    231875.000
Suburbs    194206.000
Budget     148956.000
Name: SalePrice, dtype: float64

Full neighborhood summary:
              count  mean_price  median_price  std_price  mean_size  mean_qual
Neighborhood                                                                  
Affluent         61  269114.800    274000.000  43989.800   1555.500      6.500
Midtown          72  231875.000    229500.000  47729.300   1526.200      6.700
Suburbs         102  194205.900    193000.000  41075.500   1502.200      6.500
Budget           45  148955.600    139000.000  33373.900   1476.800      6.200

=== Multi-Level GroupBy ===

                       count  mean_price
Neighborhood BldgType                   
Affluent     TwnhsE       12  274583.000
             1Fam         39  268769.000
             Duplex       10  263900.000
Midtown      TwnhsE       14  248786.000
             Duplex        9  2315

---
## Section 6 — Pandas: Apply, Map & Custom Transforms

### Three Ways to Apply Functions

| Method | What it does | Input → Output |
|---|---|---|
| `df['col'].map(func)` | Element-wise on Series | scalar → scalar |
| `df['col'].apply(func)` | Element-wise on Series | scalar → any |
| `df.apply(func, axis=1)` | Row-wise on DataFrame | row Series → scalar |
| `df.apply(func, axis=0)` | Column-wise on DataFrame | col Series → scalar |

### When to Use Each

- **`map`** — translate values: label encoding, string replacement
- **`apply` on Series** — element transform: custom log, clipping, parsing
- **`apply` on DataFrame with axis=1** — combine multiple columns per row
- **`np.vectorize`** or direct broadcasting — fastest, when possible


In [7]:
import time

np.random.seed(42)
n = 8

df = pd.DataFrame({
    'SalePrice':     [220000, 350000, 155000, 280000, 120000, 380000, 240000, 410000],
    'GrLivArea':     [1500, 2000, 1200, 1800, 900, 2200, 1600, 2400],
    'YearBuilt':     [1995, 2010, 1972, 2018, 1965, 2005, 1988, 2020],
    'Condition':     ['Normal', 'Excellent', 'Poor', 'Good', 'Poor', 'Excellent', 'Normal', 'Good'],
    'Address':       ['123 Oak St', '456 Maple Ave', '789 Pine Rd', '321 Elm Dr',
                      '654 Birch Ln', '987 Cedar Blvd', '111 Willow Way', '222 Ash Ct'],
})

print("=== map — Translate Values ===")
print()

# Encode ordinal categories as integers
condition_map = {'Poor': 1, 'Normal': 2, 'Good': 3, 'Excellent': 4}
df['ConditionNum'] = df['Condition'].map(condition_map)
print("map(condition_map):")
print(df[['Condition', 'ConditionNum']])
print()

print("=== apply on a Series — Custom Element Function ===")
print()

# Log transform with custom handling
def safe_log(x):
    if pd.isna(x) or x <= 0:
        return np.nan
    return np.log1p(x)

df['LogPrice'] = df['SalePrice'].apply(safe_log)
# Faster equivalent (prefer this when possible):
df['LogPrice2'] = np.log1p(df['SalePrice'])

print("Log-transformed prices:")
print(df[['SalePrice', 'LogPrice', 'LogPrice2']].head(4).round(3))
print(f"  (Both columns identical: {np.allclose(df['LogPrice'], df['LogPrice2'])})")
print()

# String parsing with apply
def extract_street_type(address):
    tokens = address.split()
    return tokens[-1] if tokens else 'Unknown'

df['StreetType'] = df['Address'].apply(extract_street_type)
print("Parsed street types:")
print(df[['Address', 'StreetType']])
print()

print("=== apply(axis=1) — Combine Multiple Columns Per Row ===")
print()

# Compute a feature that requires multiple columns
# (can't do this with a simple column operation)
def house_age_score(row):
    age     = 2024 - row['YearBuilt']
    quality = row['ConditionNum']
    size    = row['GrLivArea']
    # Weighted composite score
    return round(quality * 25 - age * 0.5 + size * 0.02, 1)

df['CompositeScore'] = df.apply(house_age_score, axis=1)
print("Composite score (quality + age + size):")
print(df[['YearBuilt', 'Condition', 'GrLivArea', 'CompositeScore']])
print()

print("=== Performance: When to NOT Use apply ===")
print()

n_big = 100_000
prices = pd.Series(np.random.uniform(50000, 800000, n_big))

t0 = time.perf_counter()
_ = prices.apply(lambda x: np.log1p(x))
t1 = time.perf_counter()

t2 = time.perf_counter()
_ = np.log1p(prices)
t3 = time.perf_counter()

print(f"  apply(lambda) on {n_big:,} rows: {(t1-t0)*1000:.1f} ms")
print(f"  np.log1p() vectorized:          {(t3-t2)*1000:.1f} ms   <- prefer this!")
print()
print("  Rule: if NumPy has the function, use it directly.")
print("  Use apply() only when you need Python logic (string parsing, conditions,")
print("  combining multiple columns with complex logic).")


=== map — Translate Values ===

map(condition_map):
   Condition  ConditionNum
0     Normal             2
1  Excellent             4
2       Poor             1
3       Good             3
4       Poor             1
5  Excellent             4
6     Normal             2
7       Good             3

=== apply on a Series — Custom Element Function ===

Log-transformed prices:
   SalePrice  LogPrice  LogPrice2
0     220000    12.301     12.301
1     350000    12.766     12.766
2     155000    11.951     11.951
3     280000    12.543     12.543
  (Both columns identical: True)

Parsed street types:
          Address StreetType
0      123 Oak St         St
1   456 Maple Ave        Ave
2     789 Pine Rd         Rd
3      321 Elm Dr         Dr
4    654 Birch Ln         Ln
5  987 Cedar Blvd       Blvd
6  111 Willow Way        Way
7      222 Ash Ct         Ct

=== apply(axis=1) — Combine Multiple Columns Per Row ===

Composite score (quality + age + size):
   YearBuilt  Condition  GrLivArea  Compos

---
## Section 7 — Pandas: Reshaping — Melt & Pivot

### Two Complementary Operations

| | `melt` | `pivot_table` |
|---|---|---|
| Direction | wide → long | long → wide |
| Use case | unpacking multiple score columns into one | aggregating by two categories |
| SQL analog | UNPIVOT | pivot / crosstab |

### When You'll See This

**Melt:** A dataset has `score_q1`, `score_q2`, `score_q3` columns.
You need a `quarter` column and a single `score` column for time-series analysis.

**Pivot:** You have a long table of `(neighborhood, building_type, sale_price)` 
and want to see mean price for each neighborhood × building_type combination.


In [8]:
print("=== melt — Wide to Long ===")
print()

# Typical case: multiple metric columns per row
results = pd.DataFrame({
    'model':       ['Linear', 'Ridge', 'Lasso', 'ElasticNet'],
    'train_r2':    [0.920, 0.918, 0.915, 0.917],
    'val_r2':      [0.901, 0.908, 0.906, 0.907],
    'test_r2':     [0.897, 0.905, 0.904, 0.906],
})
print("Wide format (original):")
print(results)
print()

# melt: unpivot metric columns into rows
long = results.melt(
    id_vars    = ['model'],        # columns to keep as-is
    value_vars = ['train_r2', 'val_r2', 'test_r2'],  # columns to unpivot
    var_name   = 'split',          # new column for the column names
    value_name = 'r2_score',       # new column for the values
)
print("Long format (after melt):")
print(long)
print()

# Why long format is useful: easier to plot, easier to groupby
print("Best model per split:")
print(long.groupby('split').apply(lambda g: g.loc[g['r2_score'].idxmax(), 'model']).to_frame('best_model'))
print()

print("=== pivot_table — Long to Wide (Aggregation) ===")
print()

# Typical case: one row per (neighborhood, building_type, price) transaction
np.random.seed(42)
n = 200
transactions = pd.DataFrame({
    'neighborhood': np.random.choice(['North','South','East','West'], n),
    'bldg_type':    np.random.choice(['House','Condo','Townhouse'], n),
    'sale_price':   np.random.normal(280000, 80000, n).clip(80000, 600000).round(-3),
    'sqft':         np.random.normal(1600, 400, n).clip(500, 3500).round(0),
})
print("Long format (first 5 rows):")
print(transactions.head())
print()

# pivot_table: mean price for each neighborhood × building type
pivot = transactions.pivot_table(
    values  = 'sale_price',
    index   = 'neighborhood',     # becomes rows
    columns = 'bldg_type',        # becomes columns
    aggfunc = 'mean',
    margins = True,                # adds row/col totals
).round(0)
print("pivot_table: mean sale_price by neighborhood x building type:")
print(pivot)
print()

# Count instead of mean
pivot_count = transactions.pivot_table(
    values  = 'sale_price',
    index   = 'neighborhood',
    columns = 'bldg_type',
    aggfunc = 'count',
).fillna(0).astype(int)
print("Count of transactions:")
print(pivot_count)


=== melt — Wide to Long ===

Wide format (original):
        model  train_r2  val_r2  test_r2
0      Linear     0.920   0.901    0.897
1       Ridge     0.918   0.908    0.905
2       Lasso     0.915   0.906    0.904
3  ElasticNet     0.917   0.907    0.906

Long format (after melt):
         model     split  r2_score
0       Linear  train_r2     0.920
1        Ridge  train_r2     0.918
2        Lasso  train_r2     0.915
3   ElasticNet  train_r2     0.917
4       Linear    val_r2     0.901
5        Ridge    val_r2     0.908
6        Lasso    val_r2     0.906
7   ElasticNet    val_r2     0.907
8       Linear   test_r2     0.897
9        Ridge   test_r2     0.905
10       Lasso   test_r2     0.904
11  ElasticNet   test_r2     0.906

Best model per split:
          best_model
split               
test_r2   ElasticNet
train_r2      Linear
val_r2         Ridge

=== pivot_table — Long to Wide (Aggregation) ===

Long format (first 5 rows):
  neighborhood  bldg_type  sale_price     sqft
0     

---
## Section 8 — The NumPy-Pandas Bridge

### The Central Rule

**Use Pandas** when you're manipulating tables: selecting, merging, grouping,
filling missing values, encoding categories, renaming columns.

**Use NumPy** when you're doing math: matrix multiply, linear algebra,
vectorized transforms, feeding sklearn, building neural nets.

**The handoff:**
```python
# Pandas for preprocessing
X_df = df[feature_cols].fillna(df[feature_cols].median())
X_df = pd.get_dummies(X_df, columns=['Neighborhood'])

# NumPy for math / sklearn
X = X_df.to_numpy()   # or X_df.values
y = df['SalePrice'].to_numpy()
```

### Memory & Performance Notes

- Pandas `DataFrame` stores each column as a NumPy array
- `.values` / `.to_numpy()` returns a view (no copy) when dtypes are uniform
- Mixed dtypes → forced to `object` array → slow → always clean dtypes before converting


In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, cross_val_score

np.random.seed(42)
print("=== The Full Preprocessing Pipeline ===")
print()

# Step 1: Create messy DataFrame (representative of real Kaggle data)
n = 200
df = pd.DataFrame({
    'GrLivArea':    np.where(np.random.rand(n) < 0.05, np.nan, np.random.normal(1500, 400, n).clip(400, 4000).round(0)),
    'OverallQual':  np.random.randint(3, 11, n).astype(float),
    'YearBuilt':    np.random.randint(1960, 2020, n).astype(float),
    'GarageCars':   np.where(np.random.rand(n) < 0.07, np.nan, np.random.choice([0,1,2,3], n, p=[0.05,0.2,0.6,0.15])),
    'Neighborhood': np.random.choice(['North','South','East','West'], n),
    'SalePrice':    np.random.normal(250000, 80000, n).clip(80000, 600000).round(-3),
})

print(f"Raw data: {df.shape}")
print(f"Missing: {df.isnull().sum().to_dict()}")
print()

# Step 2: Impute missing (Pandas)
df_clean = df.copy()
df_clean['GrLivArea'] = df_clean['GrLivArea'].fillna(df_clean['GrLivArea'].median())
df_clean['GarageCars'] = df_clean['GarageCars'].fillna(0)
print(f"After imputation: {df_clean.isnull().sum().sum()} missing values")

# Step 3: Feature engineering (Pandas)
df_clean['HouseAge']    = 2024 - df_clean['YearBuilt']
df_clean['QualXSize']   = df_clean['OverallQual'] * df_clean['GrLivArea']
df_clean['LogSalePrice']= np.log1p(df_clean['SalePrice'])

# Step 4: Encode categoricals (Pandas)
df_encoded = pd.get_dummies(df_clean, columns=['Neighborhood'], drop_first=True)
print(f"After one-hot encoding: {df_encoded.shape}")

# Step 5: Select feature columns
feature_cols = ['GrLivArea', 'OverallQual', 'HouseAge', 'GarageCars',
                'QualXSize'] + [c for c in df_encoded.columns if c.startswith('Neighborhood')]
target_col   = 'LogSalePrice'

# Step 6: Convert to NumPy for sklearn (THE BRIDGE)
X = df_encoded[feature_cols].to_numpy(dtype=np.float64)
y = df_encoded[target_col].to_numpy(dtype=np.float64)

print()
print("=== sklearn-ready arrays ===")
print(f"X: shape={X.shape}  dtype={X.dtype}  any_nan={np.isnan(X).any()}")
print(f"y: shape={y.shape}  dtype={y.dtype}  any_nan={np.isnan(y).any()}")
print()

# Step 7: Quick sklearn proof
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

model  = Ridge(alpha=1.0)
scores = cross_val_score(model, X_tr_s, y_tr, cv=5, scoring='r2')

model.fit(X_tr_s, y_tr)
test_r2 = model.score(X_te_s, y_te)

print("Ridge regression on engineered features:")
print(f"  CV R2 scores:   {scores.round(3)}")
print(f"  CV R2 mean:     {scores.mean():.3f}  +/- {scores.std():.3f}")
print(f"  Test R2:        {test_r2:.3f}")
print()

print("=== dtype Performance ===")
print()

df_uniform = pd.DataFrame(np.random.randn(100000, 20))
df_mixed   = df_uniform.copy()
df_mixed['cat'] = 'some_string'   # forces object dtype

t0 = time.perf_counter()
_ = df_uniform.to_numpy()
t1 = time.perf_counter()

t2 = time.perf_counter()
_ = df_mixed[df_uniform.columns].to_numpy()
t3 = time.perf_counter()

print(f"  Uniform float64 to_numpy():  {(t1-t0)*1000:.2f} ms  (view, no copy)")
print(f"  After cleaning (same data):  {(t3-t2)*1000:.2f} ms")
print()
print("  Tip: keep dtypes clean throughout the pipeline.")
print("  Run df.dtypes.value_counts() to spot accidental object columns.")


=== The Full Preprocessing Pipeline ===

Raw data: (200, 6)
Missing: {'GrLivArea': 11, 'OverallQual': 0, 'YearBuilt': 0, 'GarageCars': 10, 'Neighborhood': 0, 'SalePrice': 0}

After imputation: 0 missing values
After one-hot encoding: (200, 11)

=== sklearn-ready arrays ===
X: shape=(200, 8)  dtype=float64  any_nan=False
y: shape=(200,)  dtype=float64  any_nan=False

Ridge regression on engineered features:
  CV R2 scores:   [-0.09  -0.161 -0.078 -0.021  0.024]
  CV R2 mean:     -0.065  +/- 0.063
  Test R2:        -0.022

=== dtype Performance ===

  Uniform float64 to_numpy():  0.02 ms  (view, no copy)
  After cleaning (same data):  0.21 ms

  Tip: keep dtypes clean throughout the pipeline.
  Run df.dtypes.value_counts() to spot accidental object columns.


---
## Summary — NumPy & Pandas

### NumPy Quick Reference

```python
# Stacking
np.hstack([a, b])              # add columns
np.vstack([a, b])              # add rows
np.column_stack([v1, v2])      # 1D vectors as columns

# Splitting
np.split(arr, [50, 80], axis=0)  # rows 0-49, 50-79, 80+
X_num, X_cat = np.split(X, [3], axis=1)

# Saving
np.save('arr.npy', X)
np.savez('data.npz', X=X, y=y)
X = np.load('arr.npy')

# Indexing
X[rows, :]                     # fancy row indexing
X[:, cols]                     # fancy column indexing
X[(X[:, 0] > 0) & (y == 1)]   # boolean mask
np.where(X > 0, X, 0)         # vectorized ternary
```

### Pandas Quick Reference

```python
# Load
df = pd.read_csv('data.csv', index_col='Id')

# Merge (JOIN)
pd.merge(df1, df2, on='key', how='left')

# GroupBy
df.groupby('col').agg(mean=('target','mean'), count=('target','count'))
df.groupby('col')['target'].transform('mean')  # adds group stat per row

# Apply
df['col'].map({old: new})          # translate values
df['col'].apply(func)              # element-wise custom
df.apply(func, axis=1)             # row-wise (slower)

# Reshape
df.melt(id_vars=['id'], var_name='split', value_name='score')
df.pivot_table(values='price', index='nbhd', columns='type', aggfunc='mean')

# To sklearn
X = df[feature_cols].to_numpy(dtype=np.float64)
```

### Next Notebook
`data_cleaning.ipynb` — systematic strategies for messy real-world data
